# Getting Started with Relate

Relate is a lightweight, embeddable graph database that uses the openCypher query language. The `relate` Python package embeds the engine as a native extension module — queries execute in-process, with no subprocess or server. This notebook walks through the basics: creating nodes and relationships, querying, aggregation, graph traversal, and Arrow interop.

## Setup

Relate is not yet published to PyPI. Build the Python package from a checkout of the repository (requires the Rust toolchain):

```bash
cd surfaces/relate-python
pip install maturin
maturin develop --release
```

In [1]:
# --- Setup ------------------------------------------------------------------
# Install Relate and this notebook's dependencies into the current kernel
# if they are not already available. Relate is not yet published to PyPI,
# so it is built from this repository checkout (requires the Rust toolchain,
# https://rustup.rs) -- the same maturin build that `make install-python` runs.
# Once published, the Relate line becomes a plain `pip install relate`.
import subprocess
import sys


def _ensure(module, pip_spec):
    try:
        __import__(module)
    except ImportError:
        print(f"Installing {pip_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec])


_ensure("relate", "../surfaces/relate-python")  # local build (pre-PyPI)
_ensure("pyarrow", "pyarrow")  # Arrow IPC interop
_ensure("pandas", "pandas")  # DataFrame view of query results


In [2]:
import relate

print(f"relate version: {relate.__version__}")

relate version: 0.3.0a0


## Connect to a Database

Use `:memory:` for an in-memory database, or pass a file path for persistence.

In [3]:
db = relate.connect(":memory:")
print("Connected to in-memory database")

Connected to in-memory database


## Create Nodes

Use `CREATE` to add nodes to the graph. Parameters are passed as a dictionary with `$` references in the query.

In [4]:
people = [
    {"name": "Alice", "age": 32, "city": "Portland"},
    {"name": "Bob", "age": 28, "city": "Seattle"},
    {"name": "Charlie", "age": 35, "city": "Portland"},
    {"name": "Diana", "age": 26, "city": "San Francisco"},
    {"name": "Eve", "age": 30, "city": "Seattle"},
]

for person in people:
    cursor = db.execute(
        "CREATE (:Person {name: $name, age: $age, city: $city})",
        person,
    )

print(f"Created {len(people)} people")

Created 5 people


## Query Nodes

`db.execute()` returns a `Cursor` that you can iterate over. Each row is a dictionary.

In [5]:
cursor = db.execute(
    "MATCH (p:Person) RETURN p.name AS name, p.age AS age, p.city AS city ORDER BY p.name"
)

print(f"Columns: {cursor.columns}")
print(f"Row count: {len(cursor)}")
print()
for row in cursor:
    print(f"  {row['name']:10s}  age {row['age']}  ({row['city']})")

Columns: ['name', 'age', 'city']
Row count: 5

  Alice       age 32  (Portland)
  Bob         age 28  (Seattle)
  Charlie     age 35  (Portland)
  Diana       age 26  (San Francisco)
  Eve         age 30  (Seattle)


## Create Relationships

Use `MATCH` to find existing nodes, then `CREATE` to add relationships between them.

In [6]:
friendships = [
    ("Alice", "Bob"),
    ("Alice", "Charlie"),
    ("Bob", "Diana"),
    ("Charlie", "Eve"),
    ("Diana", "Eve"),
    ("Bob", "Eve"),
]

for a, b in friendships:
    cursor = db.execute(
        "MATCH (a:Person {name: $a}), (b:Person {name: $b}) "
        "CREATE (a)-[:FRIENDS_WITH]->(b)",
        {"a": a, "b": b},
    )

print(f"Created {len(friendships)} friendships")

Created 6 friendships


## Query Relationships

Pattern matching makes it easy to traverse the graph.

In [7]:
# Direct friends of Alice
cursor = db.execute(
    "MATCH (a:Person {name: 'Alice'})-[:FRIENDS_WITH]->(friend) "
    "RETURN friend.name AS name ORDER BY name"
)

print("Alice's friends:")
for row in cursor:
    print(f"  - {row['name']}")

Alice's friends:
  - Bob
  - Charlie


## Multi-Hop Traversal

Chain relationship patterns to traverse multiple hops.

In [8]:
# Friends-of-friends of Alice (2 hops)
cursor = db.execute(
    "MATCH (a:Person {name: 'Alice'})-[:FRIENDS_WITH]->(mid)-[:FRIENDS_WITH]->(fof) "
    "RETURN DISTINCT fof.name AS name, mid.name AS via ORDER BY name"
)

print("Alice's friends-of-friends (2 hops):")
for row in cursor:
    print(f"  - {row['name']} (via {row['via']})")

Alice's friends-of-friends (2 hops):
  - Diana (via Bob)
  - Eve (via Bob)
  - Eve (via Charlie)


## Aggregation

Use `count()`, `collect()`, `avg()`, `sum()`, and other aggregation functions.

In [9]:
# People per city
cursor = db.execute(
    "MATCH (p:Person) "
    "RETURN p.city AS city, count(p) AS total, collect(p.name) AS names "
    "ORDER BY total DESC"
)

print("People per city:")
for row in cursor:
    names = ", ".join(row["names"])
    print(f"  {row['city']:20s} {row['total']} people: {names}")

People per city:
  Portland             2 people: Alice, Charlie
  Seattle              2 people: Bob, Eve
  San Francisco        1 people: Diana


In [10]:
# Average age by city
cursor = db.execute(
    "MATCH (p:Person) "
    "RETURN p.city AS city, avg(p.age) AS avg_age, min(p.age) AS youngest, max(p.age) AS oldest "
    "ORDER BY avg_age DESC"
)

print("Age stats per city:")
for row in cursor:
    print(f"  {row['city']:20s} avg={row['avg_age']:.1f}  min={row['youngest']}  max={row['oldest']}")

Age stats per city:
  Portland             avg=33.5  min=32  max=35
  Seattle              avg=29.0  min=28  max=30
  San Francisco        avg=26.0  min=26  max=26


## Filtering with WHERE

Use parameters (`$name`) for safe, reusable queries.

In [11]:
# Find people over a given age
min_age = 30
cursor = db.execute(
    "MATCH (p:Person) WHERE p.age >= $min_age "
    "RETURN p.name AS name, p.age AS age ORDER BY p.age DESC",
    {"min_age": min_age},
)

print(f"People aged {min_age}+:")
for row in cursor:
    print(f"  {row['name']} (age {row['age']})")

People aged 30+:
  Charlie (age 35)
  Alice (age 32)
  Eve (age 30)


## UNWIND and WITH

`UNWIND` expands a list into rows. `WITH` chains query stages together.

In [12]:
# UNWIND a list of values
cursor = db.execute("UNWIND [10, 20, 30, 40, 50] AS x RETURN x, x * 2 AS doubled")

print("UNWIND example:")
for row in cursor:
    print(f"  x={row['x']}, doubled={row['doubled']}")

UNWIND example:
  x=10, doubled=20
  x=20, doubled=40
  x=30, doubled=60
  x=40, doubled=80
  x=50, doubled=100


In [13]:
# WITH for query chaining — find people, filter, then return
cursor = db.execute(
    "MATCH (p:Person) "
    "WITH p.name AS name, p.age AS age "
    "WHERE age > 28 "
    "RETURN name, age ORDER BY age"
)

print("People over 28 (using WITH):")
for row in cursor:
    print(f"  {row['name']} (age {row['age']})")

People over 28 (using WITH):
  Eve (age 30)
  Alice (age 32)
  Charlie (age 35)


## MERGE — Create If Not Exists

`MERGE` creates a node only if a matching one doesn't already exist.

In [14]:
# MERGE won't create duplicates
db.execute("MERGE (p:Person {name: 'Alice'})")
db.execute("MERGE (p:Person {name: 'Alice'})")
db.execute("MERGE (p:Person {name: 'Frank'})")

cursor = db.execute("MATCH (p:Person) RETURN p.name AS name ORDER BY name")
print("All people after MERGE (Alice not duplicated, Frank added):")
for row in cursor:
    print(f"  - {row['name']}")

All people after MERGE (Alice not duplicated, Frank added):
  - Alice
  - Bob
  - Charlie
  - Diana
  - Eve
  - Frank


## OPTIONAL MATCH

`OPTIONAL MATCH` returns `null` when no match is found, instead of filtering the row out.

In [15]:
# Frank has no friends yet
cursor = db.execute(
    "MATCH (p:Person) "
    "OPTIONAL MATCH (p)-[:FRIENDS_WITH]->(friend) "
    "RETURN p.name AS person, count(friend) AS friend_count "
    "ORDER BY friend_count DESC, person"
)

print("Friend count per person:")
for row in cursor:
    print(f"  {row['person']:10s} {row['friend_count']} friends")

Friend count per person:
  Alice      2 friends
  Bob        2 friends
  Charlie    1 friends
  Diana      1 friends
  Eve        0 friends
  Frank      0 friends


## Mutation Stats

Every `execute()` call returns a `Cursor` with a `.stats` property showing what was mutated.

In [16]:
cursor = db.execute(
    "CREATE (p:Project {name: 'Relate', language: 'Rust', stars: 0})"
)

print("Stats from CREATE:")
for key, val in cursor.stats.items():
    if val > 0:
        print(f"  {key}: {val}")

Stats from CREATE:
  nodes_created: 1
  properties_set: 3
  labels_added: 1


In [17]:
cursor = db.execute(
    "MATCH (p:Project {name: 'Relate'}) SET p.stars = 42"
)

print("Stats from SET:")
for key, val in cursor.stats.items():
    if val > 0:
        print(f"  {key}: {val}")

Stats from SET:
  properties_set: 1


## executemany — Batch Operations

Use `executemany()` to run the same query with multiple parameter sets.

In [18]:
# Batch create cities
cities = [
    {"name": "Portland", "state": "OR", "pop": 652573},
    {"name": "Seattle", "state": "WA", "pop": 749256},
    {"name": "San Francisco", "state": "CA", "pop": 873965},
]

db.executemany(
    "CREATE (:City {name: $name, state: $state, population: $pop})",
    cities,
)

cursor = db.execute(
    "MATCH (c:City) RETURN c.name AS city, c.state AS state, c.population AS pop ORDER BY pop DESC"
)

print("Cities:")
for row in cursor:
    print(f"  {row['city']}, {row['state']} — population {row['pop']:,}")

Cities:
  San Francisco, CA — population 873,965
  Seattle, WA — population 749,256
  Portland, OR — population 652,573


## fetchone / fetchall

Use `fetchone()` for a single row, `fetchall()` for all remaining rows.

In [19]:
cursor = db.execute("MATCH (p:Person) RETURN count(p) AS total")
row = cursor.fetchone()
print(f"Total people: {row['total']}")

cursor = db.execute("MATCH (p:Person) RETURN p.name AS name ORDER BY name")
all_rows = cursor.fetchall()
print(f"All names: {[r['name'] for r in all_rows]}")

Total people: 6
All names: ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank']


## Native `Database` Class and Arrow IPC

Alongside the sqlite3-style `connect()` API, the package exposes the native `Database` class directly. It returns `QueryResult` objects (with `.columns`, `.data`, and `.stats`) and adds `execute_arrow()`, which returns results as Arrow IPC bytes for zero-copy transfer to PyArrow, Pandas, or Polars.

In [20]:
from relate import Database

ndb = Database()  # in-memory; pass a file path for persistence

for person in people:
    ndb.execute(
        "CREATE (:Person {name: $name, age: $age, city: $city})",
        person,
    )

result = ndb.execute(
    "MATCH (p:Person) RETURN p.name AS name, p.age AS age ORDER BY p.name"
)
print(f"Columns: {result.columns}")
print(f"Rows:    {result.data}")

Columns: ['name', 'age']
Rows:    [{'name': 'Alice', 'age': 32}, {'name': 'Bob', 'age': 28}, {'name': 'Charlie', 'age': 35}, {'name': 'Diana', 'age': 26}, {'name': 'Eve', 'age': 30}]


In [21]:
import pyarrow as pa

# Execute and get Arrow IPC bytes
buf = ndb.execute_arrow(
    "MATCH (p:Person) RETURN p.name AS name, p.age AS age, p.city AS city ORDER BY p.name"
)

# Deserialize to a PyArrow Table, then a Pandas DataFrame
table = pa.ipc.open_stream(buf).read_all()
print(f"Arrow Table: {table.num_rows} rows, {table.num_columns} columns")

df = table.to_pandas()
print("Average age by city:")
print(df.groupby("city")["age"].mean().sort_values(ascending=False))

Arrow Table: 5 rows, 3 columns
Average age by city:
city
Portland         33.5
Seattle          29.0
San Francisco    26.0
Name: age, dtype: float64


### Database introspection

The native class provides direct access to database metadata, plus `reset()` to clear all data.

In [22]:
print(f"Node count:         {ndb.node_count()}")
print(f"Relationship count: {ndb.relationship_count()}")
print(f"Labels:             {ndb.labels()}")
print(f"Relationship types: {ndb.relationship_types()}")

ndb.reset()
print(f"After reset: {ndb.node_count()} nodes")
ndb.close()

Node count:         5
Relationship count: 0
Labels:             ['Person']
Relationship types: []
After reset: 0 nodes


## Cleanup

Always close the connection when done. You can also use a `with` statement for automatic cleanup.

In [23]:
db.close()
print(f"Connection closed: {db.closed}")

Connection closed: True


In [24]:
# Alternative: use context manager
with relate.connect(":memory:") as db:
    db.execute("CREATE (:Greeting {msg: 'Hello from Relate!'})")
    cursor = db.execute("MATCH (g:Greeting) RETURN g.msg AS msg")
    print(cursor.fetchone()["msg"])

print(f"Connection auto-closed: {db.closed}")

Hello from Relate!
Connection auto-closed: True
